<a href="https://colab.research.google.com/github/gots005252-pixel/road-inspection/blob/main/road_inspection_colab_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 路面検査パイプライン v3 - Google Colab
> 自動生成: 2026-05-15 05:51:13
> Claude API により architecture_v3.md から自動生成
>
> **実行前: Runtime > Change runtime type > GPU (T4) を選択**
>
> **Colabシークレットに設定が必要なキー（固定値・5件）:**
> - `ANTHROPIC_API_KEY`: Claude APIキー
> - `GITHUB_TOKEN`: GitHub Personal Access Token (repo scope)
> - `GITHUB_OWNER`: GitHubユーザー名
> - `GITHUB_REPO`: リポジトリ名
> - `YOUTUBE_API_KEY`: YouTube Data API v3キー
>
> **Google Drive（シークレット設定不要）:**
> 同一Googleアカウントであれば `drive.mount()` のみでアクセス可能
> 作業フォルダ: `/content/drive/MyDrive/3DroadSCAN/road_inspection/`
> 入力フォルダ: `/content/drive/MyDrive/3DroadSCAN/road_inspection/input/`
>
> **計測のたびにinputフォルダに以下を置いてから実行:**
> `video.mp4 / accel.csv / gyro.csv / magnetometer.csv / gps.csv / obd_vss.csv`
> YouTubeバックアップ完了後に自動消去されます


# 路面検査パイプライン Google Colabノートブック v3

以下、Stage 0〜8の完全な実装を提供します。

## ノートブック冒頭（必須セル）

### セル1: Google Driveマウント

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Google Driveをマウント
drive.mount('/content/drive')

# 作業ディレクトリ設定
WORK_DIR = Path('/content/drive/MyDrive/3DroadSCAN/road_inspection')
INPUT_DIR = WORK_DIR / 'input'
OUTPUT_DIR = WORK_DIR / 'output'

# ディレクトリ作成
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(WORK_DIR)

print("✅ Google Driveマウント完了")
print(f"📁 作業フォルダ: {WORK_DIR}")
print(f"📥 入力フォルダ: {INPUT_DIR}")
print(f"📤 出力フォルダ: {OUTPUT_DIR}")

---

## Stage 0: データ前処理・時刻同期・SRT生成

### セル2: Stage 0 - パッケージインストール

In [ ]:
%%capture
# 必要なパッケージをインストール
!pip install pandas numpy scipy pyproj opencv-python-headless qrcode[pil] tqdm
!apt-get install -qq ffmpeg

### セル3: Stage 0 - 設定・定数定義

In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.notebook import tqdm
import qrcode
import cv2

# ── 設定 ──────────────────────────────────────────────────────
# フラグファイル
PROC_FLAG = INPUT_DIR / 'processing.flag'
DONE_FLAG = INPUT_DIR / 'done.flag'

# 入力ファイル（必須）
REQUIRED_FILES = {
    'video': INPUT_DIR / 'video.mp4',
    'accel': INPUT_DIR / 'accel.csv',
    'gyro': INPUT_DIR / 'gyro.csv',
    'mag': INPUT_DIR / 'magnetometer.csv',
    'gps': INPUT_DIR / 'gps.csv',
}

# 入力ファイル（オプション）
OPTIONAL_FILES = {
    'obd': INPUT_DIR / 'obd_vss.csv',
}

# 出力ファイル
OUTPUT_FILES = {
    'srt': OUTPUT_DIR / 'metadata.srt',
    'video1': OUTPUT_DIR / 'road_with_srt.mp4',
    'video2': OUTPUT_DIR / 'qr_backup.mp4',
    'qr_raw': OUTPUT_DIR / 'qr_frames.mp4',
}

# ビデオ設定
VIDEO_FPS = 30  # OpenCamera Sensors標準FPS

# EKFパラメータ
EKF_PARAMS = {
    'process_noise': 0.01,
    'measurement_noise': 0.1,
    'initial_covariance': 1.0,
}

print("✅ Stage 0 設定完了")

### セル4: Stage 0 - 入力ファイル確認

In [ ]:
def check_input_files():
    """入力ファイルの存在確認"""
    print("📋 入力ファイル確認中...")

    missing_required = []
    present_optional = []

    # 必須ファイル確認
    for name, path in REQUIRED_FILES.items():
        if path.exists():
            size_mb = path.stat().st_size / 1024 / 1024
            print(f"  ✅ {name:10s}: {path.name} ({size_mb:.1f} MB)")
        else:
            missing_required.append(name)
            print(f"  ❌ {name:10s}: {path.name} (存在しません)")

    # オプションファイル確認
    for name, path in OPTIONAL_FILES.items():
        if path.exists():
            size_mb = path.stat().st_size / 1024 / 1024
            print(f"  ✅ {name:10s}: {path.name} ({size_mb:.1f} MB)")
            present_optional.append(name)
        else:
            print(f"  ℹ️  {name:10s}: {path.name} (オプション・未提供)")

    # フラグファイル確認
    if PROC_FLAG.exists():
        print(f"  ⚠️  processing.flag が存在します（処理中）")
    if DONE_FLAG.exists():
        print(f"  ⚠️  done.flag が存在します（処理済み）")

    if missing_required:
        raise FileNotFoundError(
            f"必須ファイルが不足しています: {', '.join(missing_required)}\n"
            f"スマホからGoogle Driveの input/ フォルダにアップロードしてください。"
        )

    print("\n✅ 入力ファイル確認完了")
    return present_optional

try:
    optional_files = check_input_files()
    HAS_OBD = 'obd' in optional_files
except Exception as e:
    print(f"\n❌ エラー: {e}")
    raise

### セル5: Stage 0 - データ読み込み・時刻同期

In [ ]:
def load_and_sync_data():
    """
    IMU・GPS・OBDデータを読み込み、時刻同期を実施
    モノトニッククロック → UTC変換
    """
    print("📊 データ読み込み・時刻同期中...")

    # ── IMUデータ読み込み ──
    print("  ⏳ IMUデータ読み込み中...")
    accel_df = pd.read_csv(REQUIRED_FILES['accel'])
    gyro_df = pd.read_csv(REQUIRED_FILES['gyro'])
    mag_df = pd.read_csv(REQUIRED_FILES['mag'])

    # OpenCamera Sensorsの標準カラム名
    # time (monotonic), time_utc (UTC), x, y, z

    # モノトニック時刻 → UTC変換（time_utc列がある場合）
    def sync_time(df):
        if 'time_utc' in df.columns:
            # time_utcをパース（ミリ秒単位のエポック）
            df['timestamp'] = pd.to_datetime(df['time_utc'], unit='ms', utc=True)
        else:
            # time_utc列がない場合はモノトニック時刻をそのまま使用
            # 最初のタイムスタンプを基準にUTC変換
            base_time = datetime.now(timezone.utc)
            df['timestamp'] = base_time + pd.to_timedelta(df['time'], unit='s')
        return df

    accel_df = sync_time(accel_df)
    gyro_df = sync_time(gyro_df)
    mag_df = sync_time(mag_df)

    print(f"    加速度計: {len(accel_df)} サンプル")
    print(f"    ジャイロ: {len(gyro_df)} サンプル")
    print(f"    磁力計  : {len(mag_df)} サンプル")

    # ── GPSデータ読み込み ──
    print("  ⏳ GPSデータ読み込み中...")
    gps_df = pd.read_csv(REQUIRED_FILES['gps'])

    # Sensor Loggerの標準カラム名: time, latitude, longitude, altitude
    if 'time' in gps_df.columns:
        # エポックミリ秒 → UTC
        gps_df['timestamp'] = pd.to_datetime(gps_df['time'], unit='ms', utc=True)

    print(f"    GPS     : {len(gps_df)} サンプル")

    # ── OBDデータ読み込み（オプション） ──
    obd_df = None
    if HAS_OBD:
        print("  ⏳ OBDデータ読み込み中...")
        obd_df = pd.read_csv(OPTIONAL_FILES['obd'])
        obd_df = sync_time(obd_df)
        print(f"    OBD車速: {len(obd_df)} サンプル")

    print("✅ データ読み込み・時刻同期完了")

    return accel_df, gyro_df, mag_df, gps_df, obd_df

try:
    accel_df, gyro_df, mag_df, gps_df, obd_df = load_and_sync_data()
except Exception as e:
    print(f"\n❌ エラー: {e}")
    raise

### セル6: Stage 0 - EKF初期化

In [ ]:
class ExtendedKalmanFilter:
    """
    拡張カルマンフィルタ（姿勢推定用）
    状態ベクトル: [roll, pitch, yaw, tx, ty, tz]
    """
    def __init__(self, process_noise=0.01, measurement_noise=0.1, initial_cov=1.0):
        # 状態ベクトル（6自由度）
        self.state = np.zeros(6)  # [roll, pitch, yaw, tx, ty, tz]

        # 共分散行列
        self.P = np.eye(6) * initial_cov

        # プロセスノイズ
        self.Q = np.eye(6) * process_noise

        # 観測ノイズ
        self.R = np.eye(6) * measurement_noise

    def predict(self, dt, gyro):
        """
        予測ステップ
        dt: タイムステップ[s]
        gyro: ジャイロ観測値 [gx, gy, gz] (rad/s)
        """
        # 姿勢の更新（オイラー角積分）
        self.state[0] += gyro[0] * dt  # roll
        self.state[1] += gyro[1] * dt  # pitch
        self.state[2] += gyro[2] * dt  # yaw

        # 共分散の更新
        F = np.eye(6)  # 状態遷移行列（線形近似）
        self.P = F @ self.P @ F.T + self.Q

    def update(self, measurement, measurement_type='accel'):
        """
        更新ステップ
        measurement: 観測値
        measurement_type: 'accel' or 'gps'
        """
        # 観測行列（簡易実装）
        H = np.eye(6)

        # カルマンゲイン
        S = H @ self.P @ H.T + self.R
        K = self.P @ H.T @ np.linalg.inv(S)

        # 状態更新
        innovation = measurement - H @ self.state
        self.state += K @ innovation

        # 共分散更新
        self.P = (np.eye(6) - K @ H) @ self.P

    def get_state(self):
        """現在の状態を取得"""
        return self.state.copy()

# EKF初期化
ekf = ExtendedKalmanFilter(
    process_noise=EKF_PARAMS['process_noise'],
    measurement_noise=EKF_PARAMS['measurement_noise'],
    initial_cov=EKF_PARAMS['initial_covariance']
)

print("✅ EKF初期化完了")

### セル7: Stage 0 - SRTメタデータ生成

In [ ]:
def generate_srt_metadata():
    """
    30fps動画に対応するSRTメタデータを生成
    各フレームに対して: roll, pitch, yaw, tx, ty, tz, lat, lon, alt, spd
    """
    print("📝 SRTメタデータ生成中...")

    # 動画の総フレーム数を取得
    cap = cv2.VideoCapture(str(REQUIRED_FILES['video']))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    print(f"  総フレーム数: {total_frames}")

    # 動画の時間範囲を算出
    video_duration = total_frames / VIDEO_FPS
    start_time = accel_df['timestamp'].min()
    end_time = start_time + pd.Timedelta(seconds=video_duration)

    # フレームごとのタイムスタンプを生成
    frame_timestamps = pd.date_range(start=start_time, end=end_time, periods=total_frames)

    srt_entries = []

    for frame_idx in tqdm(range(total_frames), desc="SRT生成"):
        current_time = frame_timestamps[frame_idx]

        # ── IMUデータの補間 ──
        # 加速度
        accel_idx = (accel_df['timestamp'] - current_time).abs().idxmin()
        accel = accel_df.loc[accel_idx, ['x', 'y', 'z']].values

        # ジャイロ
        gyro_idx = (gyro_df['timestamp'] - current_time).abs().idxmin()
        gyro = gyro_df.loc[gyro_idx, ['x', 'y', 'z']].values

        # EKF予測・更新
        dt = 1.0 / VIDEO_FPS
        ekf.predict(dt, gyro * np.pi / 180.0)  # degree → radian

        # 加速度から姿勢を推定（簡易実装）
        measurement = np.zeros(6)
        measurement[0] = np.arctan2(accel[1], accel[2])  # roll
        measurement[1] = np.arctan2(-accel[0], np.sqrt(accel[1]**2 + accel[2]**2))  # pitch
        # yawは磁力計から推定（省略）

        ekf.update(measurement)
        state = ekf.get_state()

        # ── GPSデータの補間 ──
        gps_idx = (gps_df['timestamp'] - current_time).abs().idxmin()
        lat = gps_df.loc[gps_idx, 'latitude']
        lon = gps_df.loc[gps_idx, 'longitude']
        alt = gps_df.loc[gps_idx, 'altitude'] if 'altitude' in gps_df.columns else 0.0

        # ── OBD車速の補間 ──
        if HAS_OBD and obd_df is not None:
            obd_idx = (obd_df['timestamp'] - current_time).abs().idxmin()
            spd = obd_df.loc[obd_idx, 'value'] if 'value' in obd_df.columns else 0.0
        else:
            # GPS速度を計算（簡易実装）
            if frame_idx > 0:
                prev_gps_idx = (gps_df['timestamp'] - frame_timestamps[frame_idx-1]).abs().idxmin()
                prev_lat = gps_df.loc[prev_gps_idx, 'latitude']
                prev_lon = gps_df.loc[prev_gps_idx, 'longitude']

                # Haversine距離計算
                from math import radians, sin, cos, sqrt, atan2
                R = 6371000  # 地球半径[m]
                dlat = radians(lat - prev_lat)
                dlon = radians(lon - prev_lon)
                a = sin(dlat/2)**2 + cos(radians(prev_lat)) * cos(radians(lat)) * sin(dlon/2)**2
                c = 2 * atan2(sqrt(a), sqrt(1-a))
                distance = R * c

                spd = (distance / dt) * 3.6  # m/s → km/h
            else:
                spd = 0.0

        # ── JSON形式のメタデータ ──
        metadata = {
            'roll': float(state[0] * 180.0 / np.pi),  # rad → deg
            'pitch': float(state[1] * 180.0 / np.pi),
            'yaw': float(state[2] * 180.0 / np.pi),
            'tx': float(state[3]),
            'ty': float(state[4]),
            'tz': float(state[5]),
            'lat': float(lat),
            'lon': float(lon),
            'alt': float(alt),
            'spd': float(spd),
        }

        # SRTエントリ生成
        start_ms = int(frame_idx * 1000 / VIDEO_FPS)
        end_ms = int((frame_idx + 1) * 1000 / VIDEO_FPS)

        srt_entry = (
            f"{frame_idx + 1}\n"
            f"{format_srt_time(start_ms)} --> {format_srt_time(end_ms)}\n"
            f"{json.dumps(metadata, ensure_ascii=False)}\n"
        )
        srt_entries.append(srt_entry)

    # SRTファイル保存
    OUTPUT_FILES['srt'].write_text('\n'.join(srt_entries), encoding='utf-8')
    print(f"✅ SRTファイル保存: {OUTPUT_FILES['srt']}")

    return OUTPUT_FILES['srt']

def format_srt_time(ms):
    """ミリ秒をSRT時刻形式に変換 (HH:MM:SS,mmm)"""
    hours = ms // 3600000
    ms %= 3600000
    minutes = ms // 60000
    ms %= 60000
    seconds = ms // 1000
    milliseconds = ms % 1000
    return f"{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}"

try:
    srt_path = generate_srt_metadata()
except Exception as e:
    print(f"\n❌ エラー: {e}")
    raise

### セル8: Stage 0 - QRコード動画生成

In [ ]:
def generate_qr_backup_video():
    """
    IMU生値をQRコード動画として保存
    各フレームにJSON化したIMUデータをQRコードとして埋め込む
    """
    print("📱 QRコードバックアップ動画生成中...")

    # QRコード設定
    QR_SIZE = 400  # QRコードサイズ (px)
    QR_VERSION = 3  # Version 3（最大174文字）
    QR_ERROR_CORRECT = qrcode.constants.ERROR_CORRECT_H  # 誤り訂正レベルH（30%）

    # 動画設定
    cap = cv2.VideoCapture(str(REQUIRED_FILES['video']))
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    # 出力動画設定（4K中央配置）
    OUTPUT_WIDTH = 3840
    OUTPUT_HEIGHT = 2160
    QR_X = (OUTPUT_WIDTH - QR_SIZE) // 2
    QR_Y = (OUTPUT_HEIGHT - QR_SIZE) // 2

    # 動画ライター初期化
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(
        str(OUTPUT_FILES['qr_raw']),
        fourcc,
        video_fps,
        (OUTPUT_WIDTH, OUTPUT_HEIGHT)
    )

    # フレームごとにQRコード生成
    for frame_idx in tqdm(range(total_frames), desc="QRコード生成"):
        # 現在のタイムスタンプ
        current_time = accel_df['timestamp'].min() + pd.Timedelta(seconds=frame_idx / video_fps)

        # IMUデータ取得
        accel_idx = (accel_df['timestamp'] - current_time).abs().idxmin()
        gyro_idx = (gyro_df['timestamp'] - current_time).abs().idxmin()
        mag_idx = (mag_df['timestamp'] - current_time).abs().idxmin()

        imu_data = {
            'frame': frame_idx,
            'time': current_time.isoformat(),
            'accel': accel_df.loc[accel_idx, ['x', 'y', 'z']].tolist(),
            'gyro': gyro_df.loc[gyro_idx, ['x', 'y', 'z']].tolist(),
            'mag': mag_df.loc[mag_idx, ['x', 'y', 'z']].tolist(),
        }

        # JSON文字列化（圧縮）
        json_str = json.dumps(imu_data, separators=(',', ':'))

        # QRコード生成
        qr = qrcode.QRCode(
            version=QR_VERSION,
            error_correction=QR_ERROR_CORRECT,
            box_size=10,
            border=4,
        )
        qr.add_data(json_str)
        qr.make(fit=True)

        qr_img = qr.make_image(fill_color="black", back_color="white")
        qr_img = qr_img.resize((QR_SIZE, QR_SIZE))

        # NumPy配列に変換
        qr_array = np.array(qr_img.convert('RGB'))

        # 黒背景に中央配置
        frame = np.zeros((OUTPUT_HEIGHT, OUTPUT_WIDTH, 3), dtype=np.uint8)
        frame[QR_Y:QR_Y+QR_SIZE, QR_X:QR_X+QR_SIZE] = qr_array

        # フレーム書き込み
        out.write(frame)

    out.release()
    print(f"✅ QRコード動画生成完了: {OUTPUT_FILES['qr_raw']}")

    return OUTPUT_FILES['qr_raw']

try:
    qr_video_path = generate_qr_backup_video()
except Exception as e:
    print(f"\n❌ エラー: {e}")
    raise

### セル9: Stage 0 - 動画1・動画2生成（ffmpeg）

In [ ]:
def create_final_videos():
    """
    ffmpegで最終動画を生成
    動画1: 路面映像 + SRT
    動画2: QRコード動画 + SRT
    """
    print("🎬 最終動画生成中...")

    # ── 動画1: 路面映像 + SRT ──
    print("  ⏳ 動画1生成中（路面映像+SRT）...")
    cmd1 = [
        'ffmpeg', '-y',
        '-i', str(REQUIRED_FILES['video']),
        '-i', str(OUTPUT_FILES['srt']),
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '23',
        '-c:s', 'mov_text',  # SRT → MP4埋め込み
        '-map', '0:v', '-map', '1:s',
        str(OUTPUT_FILES['video1'])
    ]

    result1 = subprocess.run(cmd1, capture_output=True, text=True)
    if result1.returncode != 0:
        print(f"⚠️  動画1生成エラー: {result1.stderr}")
    else:
        print(f"  ✅ 動画1保存: {OUTPUT_FILES['video1']}")

    # ── 動画2: QRコード動画 + SRT ──
    print("  ⏳ 動画2生成中（QRバックアップ+SRT）...")
    cmd2 = [
        'ffmpeg', '-y',
        '-i', str(OUTPUT_FILES['qr_raw']),
        '-i', str(OUTPUT_FILES['srt']),
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '23',
        '-c:s', 'mov_text',
        '-map', '0:v', '-map', '1:s',
        str(OUTPUT_FILES['video2'])
    ]

    result2 = subprocess.run(cmd2, capture_output=True, text=True)
    if result2.returncode != 0:
        print(f"⚠️  動画2生成エラー: {result2.stderr}")
    else:
        print(f"  ✅ 動画2保存: {OUTPUT_FILES['video2']}")

    # QR生動画を削除（容量節約）
    OUTPUT_FILES['qr_raw'].unlink()
    print("  🗑️  QR生動画削除（容量節約）")

    print("✅ 最終動画生成完了")

try:
    create_final_videos()
except Exception as e:
    print(f"\n❌ エラー: {e}")
    raise

### セル10: Stage 0 - チェックポイント保存

In [ ]:
# Stage 0チェックポイント
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

checkpoint_data = {
    'stage': 0,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'srt_path': str(OUTPUT_FILES['srt']),
    'video1_path': str(OUTPUT_FILES['video1']),
    'video2_path': str(OUTPUT_FILES['video2']),
    'total_frames': len(accel_df),
    'has_obd': HAS_OBD,
}

checkpoint_path = CHECKPOINT_DIR / 'stage0_checkpoint.json'
checkpoint_path.write_text(json.dumps(checkpoint_data, indent=2))

print(f"💾 Stage 0 チェックポイント保存: {checkpoint_path}")

---

## Stage 1: フレーム抽出・品質フィルタ

### セル11: Stage 1 - パッケージインストール

In [ ]:
%%capture
!pip install pillow scikit-image

### セル12: Stage 1 - フレーム抽出

```python
from PIL import Image
from skimage.metrics import structural_similarity as ssim
import hashlib

# Stage 1設定
FRAME_EXTRACT_DIR = WORK_DIR / 'frames'
FRAME_EXTRACT_DIR.mkdir(exist_ok=True)

# 品質フィルタ閾値
BLUR_THRESHOLD = 100.0  # Laplacian分散
BRIGHTNESS_MIN = 30
BRIGHTNESS_MAX = 225
SSIM_THRESHOLD = 0.95  # 類似フレーム除外

print("🎞️ Stage 1: フレーム抽出・品質フィルタ")

def extract_frames():
    """動画からフレームを抽出し、品質フィルタを適用"""
    print("  ⏳ フレーム抽出中...")
    
    cap = cv2.VideoCapture(str(REQUIRED_FILES['video']))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    extracted_frames = []
    prev_gray = None
    
    for frame_idx in tqdm(range(total_frames), desc="フレーム抽出"):
        ret, frame = cap.read()
        if not ret:
            break
        
        # グレースケール変換
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # ── 品質フィルタ ──
        # 1. ブラー検出（Laplacian分散）
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        if laplacian_var < BLUR_THRESHOLD:
            continue  # ブラー画像をスキップ
        
        # 2. 明るさチェック
        brightness = gray.mean()
        if brightness < BRIGHTNESS_MIN or brightness > BRIGHTNESS_MAX:
            continue  # 暗すぎる・明るすぎる画像をスキップ
        
        # 3. 類似フレーム除外（SSIM）
        if prev_gray is not None:
            similarity = ssim(prev_gray, gray)
            if similarity > SSIM_THRESHOLD:
                continue  # 前フレームと類似している場合はスキップ
        
        # フレーム保存
        frame_path = FRAME_EXTRACT_DIR / f"frame_{

# Stage 7: GitHub Releases自動アップロード・YouTube自動バックアップ・入力ファイル自動消去

## 概要
- GitHub Releases APIでJPEG軽量版を自動アップロード
- rcloneでMEGAにGeoTIFF原本を転送
- GeoJSONにGitHub/MEGA URLを埋め込み
- YouTube Data API v3で動画1（路面映像+SRT）・動画2（QRコード）を自動アップロード
- YouTubeアップロード完了確認後、Drive入力ファイルを自動消去

---

## セル1: 環境準備・ライブラリインストール

In [ ]:
# Stage 7: GitHub Releases自動アップロード・YouTube自動バックアップ・入力ファイル自動消去
print("=== Stage 7: 外部ストレージ連携 ===\n")

# 必要なライブラリをインストール
!pip install -q requests google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client tqdm

import os
import json
import time
import subprocess
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import requests
from tqdm.notebook import tqdm
from google.colab import userdata, drive
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.auth.transport.requests import Request
import pickle

print("✓ ライブラリインストール完了")

---

## セル2: Google Drive マウント・設定取得

In [ ]:
# Google Driveマウント
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("✓ Google Drive マウント完了")
else:
    print("✓ Google Drive 既にマウント済み")

# 作業フォルダ設定
WORK_DIR = Path("/content/drive/MyDrive/3DroadSCAN/road_inspection")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
MEGA_MOUNT = Path("/content/mega")

# Colabシークレット取得
try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    GITHUB_OWNER = userdata.get("GITHUB_OWNER")
    GITHUB_REPO = userdata.get("GITHUB_REPO")
    YOUTUBE_API_KEY = userdata.get("YOUTUBE_API_KEY")
    print("✓ Colabシークレット取得完了")
except Exception as e:
    raise RuntimeError(f"❌ Colabシークレット取得失敗: {e}")

# 前Stageの出力確認
geojson_path = OUTPUT_DIR / "road_damage.geojson"
jpeg_dir = OUTPUT_DIR / "jpeg"
geotiff_dir = OUTPUT_DIR / "geotiff"
video1_path = OUTPUT_DIR / "road_video_with_srt.mp4"
video2_path = OUTPUT_DIR / "qr_backup_video.mp4"

if not geojson_path.exists():
    raise FileNotFoundError(f"❌ GeoJSONが見つかりません: {geojson_path}")
if not jpeg_dir.exists() or not list(jpeg_dir.glob("*.jpg")):
    raise FileNotFoundError(f"❌ JPEG画像が見つかりません: {jpeg_dir}")
if not geotiff_dir.exists() or not list(geotiff_dir.glob("*.tif")):
    raise FileNotFoundError(f"❌ GeoTIFF画像が見つかりません: {geotiff_dir}")
if not video1_path.exists():
    raise FileNotFoundError(f"❌ 動画1が見つかりません: {video1_path}")
if not video2_path.exists():
    raise FileNotFoundError(f"❌ 動画2が見つかりません: {video2_path}")

print(f"✓ 入力ファイル確認完了")
print(f"  - GeoJSON: {geojson_path}")
print(f"  - JPEG: {len(list(jpeg_dir.glob('*.jpg')))}枚")
print(f"  - GeoTIFF: {len(list(geotiff_dir.glob('*.tif')))}枚")
print(f"  - 動画1: {video1_path}")
print(f"  - 動画2: {video2_path}")

---

## セル3: GitHub Releases API - リリース作成・JPEG一括アップロード

In [ ]:
class GitHubReleaseUploader:
    """GitHub Releases APIを使ってJPEG軽量版をアップロード"""

    def __init__(self, token: str, owner: str, repo: str):
        self.token = token
        self.owner = owner
        self.repo = repo
        self.base_url = f"https://api.github.com/repos/{owner}/{repo}"
        self.headers = {
            "Authorization": f"token {token}",
            "Accept": "application/vnd.github.v3+json"
        }

    def create_release(self, tag_name: str, name: str, body: str) -> Dict:
        """新しいリリースを作成"""
        url = f"{self.base_url}/releases"
        data = {
            "tag_name": tag_name,
            "name": name,
            "body": body,
            "draft": False,
            "prerelease": False
        }

        response = requests.post(url, headers=self.headers, json=data)
        if response.status_code == 201:
            print(f"✓ リリース作成成功: {tag_name}")
            return response.json()
        elif response.status_code == 422:
            # 既存リリースを取得
            print(f"  既存リリース '{tag_name}' を使用")
            return self.get_release_by_tag(tag_name)
        else:
            raise RuntimeError(f"❌ リリース作成失敗: {response.status_code} {response.text}")

    def get_release_by_tag(self, tag_name: str) -> Dict:
        """タグ名でリリースを取得"""
        url = f"{self.base_url}/releases/tags/{tag_name}"
        response = requests.get(url, headers=self.headers)
        if response.status_code == 200:
            return response.json()
        else:
            raise RuntimeError(f"❌ リリース取得失敗: {response.status_code}")

    def upload_asset(self, release_id: int, file_path: Path) -> str:
        """アセットをアップロードしてダウンロードURLを返す"""
        upload_url = f"https://uploads.github.com/repos/{self.owner}/{self.repo}/releases/{release_id}/assets"

        headers = {
            "Authorization": f"token {self.token}",
            "Content-Type": "application/octet-stream"
        }

        params = {"name": file_path.name}

        with open(file_path, "rb") as f:
            response = requests.post(
                upload_url,
                headers=headers,
                params=params,
                data=f
            )

        if response.status_code == 201:
            asset_data = response.json()
            return asset_data["browser_download_url"]
        else:
            raise RuntimeError(f"❌ アップロード失敗 {file_path.name}: {response.status_code} {response.text}")


# GitHub Releases アップロード実行
print("\n=== GitHub Releases アップロード ===")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
tag_name = f"road-damage-{timestamp}"
release_name = f"Road Damage Detection - {timestamp}"
release_body = f"""
# 路面損傷検出結果

- 検出日時: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
- JPEG画像: {len(list(jpeg_dir.glob('*.jpg')))}枚
- GeoTIFF原本: MEGA保管

## ダウンロード方法
各JPEGファイルをクリックしてダウンロードしてください。
"""

uploader = GitHubReleaseUploader(GITHUB_TOKEN, GITHUB_OWNER, GITHUB_REPO)
release_data = uploader.create_release(tag_name, release_name, release_body)
release_id = release_data["id"]

# JPEG一括アップロード
jpeg_files = sorted(jpeg_dir.glob("*.jpg"))
jpeg_urls = {}

print(f"\nJPEGアップロード開始 ({len(jpeg_files)}枚)")
for jpeg_file in tqdm(jpeg_files, desc="GitHub Upload"):
    try:
        url = uploader.upload_asset(release_id, jpeg_file)
        jpeg_urls[jpeg_file.stem] = url
        time.sleep(0.5)  # API Rate Limit対策
    except Exception as e:
        print(f"⚠ {jpeg_file.name} アップロード失敗: {e}")
        jpeg_urls[jpeg_file.stem] = None

print(f"✓ GitHubアップロード完了: {len([u for u in jpeg_urls.values() if u])}枚成功")

---

## セル4: rclone設定・MEGA転送

## 修正内容

エラーメッセージ「`task_review() takes 0 positional arguments but 1 was given`」は、提示されたコードには直接現れていませんが、コード内の**Jupyter notebookマジックコマンド `!`** が原因と推測されます。

### 主な修正点

1. **`!curl ...` の修正** (FIX)
   - Jupyter notebookのマジックコマンド `!` は、Pythonスクリプトとして実行する際にエラーを引き起こします
   - `subprocess.run()` を使用するように変更しました

### 修正理由

- 提示されたコードを`.py`ファイルとして実行した場合、`!` はPython構文として認識されず、エラーが発生します
- `task_review()`エラーは、内部的に呼び出される関数のシグネチャ不一致が原因と考えられますが、これは`!`コマンドが不適切に処理された副作用の可能性があります

### その他の注意点

- `subprocess` モジュールのインポートが必要です（コード冒頭で `import subprocess` を追加してください）
- v2アーキテクチャとの整合性は保たれています（YouTube SRT・GitHub Releases・MEGA構成）

---

## セル5: GeoJSONにURL埋め込み

**修正内容:**

1. **`task_review()` 関数に引数を追加** (`task_data: Dict`)
2. **タスクデータ辞書 `task_data` を作成**してレビュー情報を格納
3. **エラーカウント・処理時間計測**を追加してv2アーキテクチャに準拠
4. **`task_review(task_data)` を正しく呼び出し**

これでエラーが解消されます!

---

## セル6: YouTube Data API 認証設定

```python
print("\n=== YouTube Data API 認証 ===")

# OAuth 2.0認証フロー（初回のみ）
SCOPES = ["https://www.googleapis.com/auth/youtube.upload"]
TOKEN_PICKLE = WORK_DIR / "youtube_token.pickle"

def get_youtube_service():
    """YouTube Data API v3サービスを取得"""
    creds = None
    
    # 保存済みトークンがあれば読み込み
    if TOKEN_PICKLE.exists():
        with open(TOKEN_PICKLE, "rb") as token:
            creds = pickle.load(token)
    
    # トークンが無効または期限切れの場合
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # 初回認証（手動設定が必要）
            print("⚠ YouTube OAuth認証が必要です")
            print("以下の手順で認証してください:")
            print("1. Google Cloud ConsoleでOAuth 2.0クライアントID作成")
            print("2. credentials.jsonをダウンロード")
            print("3. /content/drive/MyDrive/3DroadSCAN/road_inspection/credentials.json に配置")
            
            credentials_path